In [ ]:
# ruff: noqa: T201, T203, E402
from __future__ import annotations

import logging
import os
import shutil
from datetime import datetime
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

while Path.cwd().name in ["notebooks", "evaluation"]:
    os.chdir("..")

from config.config import METRICS_DIR, OUTPUTS_DIR, RUN_DIR
from config.runs_vars import RUNS_METADATA, SUBSET_TO_DATASET_MAP

plt.rcParams["figure.dpi"] = 300

In [ ]:
from functools import partial

from src.analysis import (
    get_experiments_metrics,
    make_radar_chart,
    plot_detailed_analysis,
    plot_heatmap,
    plot_nfew_shots_analysis_mean,
    preprocess_metrics_df,
)

In [ ]:
PROMPT_NAMES_MAP = {
    "direct_entity_ontological": "P",
    "sequential_hierarchy_ontological": "P_{EC}",
    "direct_entity": "P^{NLF}",
    "sequential_hierarchy": "P_{EC}^{NLF}",
    "direct_entity_with_synonyms": "P_{S}^{NLF}",
    "sequential_hierarchy_with_synonyms": "P_{EC+S}^{NLF}",
    # "all_data_dummy": "P_{A}^{D}",
    # "only_names": "P_{N}^{D}",
    # "only_with_children": "P_{C}^{D}",
    # "only_with_parents": "P_{P}^{D}",
    # "with_hierarchy": "P_{H}^{D}",
}

MODEL_NAMES_MAP = {
    "gemini-1.5-flash": "Flash.1.5",
    "gemini-2.0-flash": "Flash.2.0",
    "gemini-2.0-flash-lite": "Flash.2.0.lite",
    "gemini-2.5-flash-preview-04-17": "Flash.2.5",
    "gpt-4o-mini": "gpt.4o.mini",
}

TASK_NAMES_MAP = {
    x: x
    for x in [
        "human-mouse",
        "ncit-doid",
        "omim-ordo",
        "snomed-fma.body",
        "snomed-ncit.neoplas",
        "snomed-ncit.pharm",
        "fma-nci",
        "fma-snomed",
        "snomed-nci",
    ]
}

# bind notebook-specific display maps to the shared src implementation
preprocess_metrics_df = partial(
    preprocess_metrics_df,
    prompt_names_map=PROMPT_NAMES_MAP,
    model_names_map=MODEL_NAMES_MAP,
    task_names_map=TASK_NAMES_MAP,
)

## Detailed plotting of the experiment results

In [ ]:
suffix = ""
metrics_df = preprocess_metrics_df(pd.read_csv(RUN_DIR / f"all_runs_metrics{suffix}.csv"))

metrics_df.head(7)
metrics_df["Prompt"].unique()

In [ ]:
metrics_df

In [ ]:
metrics_df = get_experiments_metrics(
    ["human-mouse", "ncit-doid", "omim-ordo", "fma-nci"],
    ["gemini-2.0-flash"],
    {"determinism": ["0", "1", "2", "3"]},
    suffix="",
)
# metrics_df = get_experiments_metrics(
#     ["human-mouse", "ncit-doid", "omim-ordo", "fma-nci"],
#     ["gemini-2.0-flash"],
#     {"syspromts": ["base", "natural_language", "none", "ontology_aware", "synonym_aware"]},
#     suffix="",
# )
metrics_df = preprocess_metrics_df(metrics_df)
metrics_df.head()

In [ ]:
experiment_col = "ExpName"
run_col_name = "Prompt"

val_col = "Youden's index"

ax = plt.subplots(figsize=(12, 6))[1]
ax.set_title(f"{val_col} Heatmap: Prompt vs Sysprompt", fontsize=18, pad=20)
# add padding to the title

plot_heatmap(
    data=metrics_df,
    row_col=left_col,
    col_col=experiment_col,
    val_col=val_col,
    precision=4,
    ax=ax,
)

In [ ]:
metrics_df.head()

In [ ]:
left_col = "Prompt"
right_col = "ExpName"
experiment_col = "Matching Task"
run_col_name = "Prompt-Execution"

val_col = "Youden's index"

metrics_df[run_col_name] = metrics_df[left_col].str.cat(metrics_df[right_col], sep="-")

ax = plt.subplots(figsize=(15, 10))[1]
ax.set_title(f"{val_col} Heatmap: {run_col_name} vs {experiment_col}", fontsize=20, pad=20)
# add padding to the title

plot_heatmap(
    data=metrics_df,
    row_col=run_col_name,
    col_col=experiment_col,
    val_col=val_col,
    precision=4,
    ax=ax,
)

In [ ]:
experiment_col, left_col, right_col = "Matching Task", "Model", "Prompt"

for suffix in ["", "_reduced"]:
    for metric in ["Youden's index", "F1 Score"]:
        metrics_df = preprocess_metrics_df(pd.read_csv(RUN_DIR / f"all_runs_metrics{suffix}.csv"))
        plot_detailed_analysis(
            metrics_df=metrics_df,
            metric=metric,
            run_col_name="LLM-Prompt",
            experiment_col=experiment_col,
            left_col=left_col,
            right_col=right_col,
            left_col_name="Model",
            right_col_name="Prompt",
            run_separator="-",
            precision=3,
            subfolder_name=f"{metric}{suffix}",
            save_plots=False,
)
        plt.show()

## Copy best results

In [ ]:
prompt_type = "direct_entity_with_synonyms---"
model = "gemini-2.5-flash-preview-04-17"

In [ ]:
for run_metadata in RUNS_METADATA:
    run_name, model_name, subset_name, experiment_name = run_metadata
    run_dir = Path("runs") / run_name
    for file in run_dir.iterdir():
        if prompt_type in str(file) and model in str(file) and str(file).endswith(".csv"):
            logging.info(f"Processing {file} in {file}")
            new_file_path = OUTPUTS_DIR / "best_results" / datetime.now().strftime("%Y-%m-%d") / file.name
            new_file_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(file, new_file_path)

## Gather outputs inference

In [ ]:
from collections import defaultdict
import json

base_dir = Path("/home/sviat/code/rai/KG-LLM-Matching/outputs/anatomy/human-mouse")
models_folders = [d.name for d in base_dir.iterdir() if d.is_dir()]


model_to_experiments = defaultdict(dict)
for model in models_folders:
    model_dir = base_dir / model
    for exp_type_dir in model_dir.iterdir():
        if not exp_type_dir.is_dir():
            continue
        experiment_names = []
        for exp_dir in exp_type_dir.iterdir():
            if exp_dir.is_dir():
                # continue
                experiment_names.append(exp_dir.name)
        model_to_experiments[model][exp_type_dir.name] = experiment_names

dict(model_to_experiments)
with open("model_to_experiments.json", "w") as f:
    json.dump(model_to_experiments, f, indent=4)

In [ ]:
all_models = set()
all_experiments: dict[str, set] = {}
all_tasks = [
    "human-mouse",
    # "ncit-doid",
    # "omim-ordo",
    # "snomed-fma.body",
    # "snomed-ncit.neoplas",
    # "snomed-ncit.pharm",
    # "fma-nci",
    # "fma-snomed",
    # "snomed-nci",
]

for model, experiments in model_to_experiments.items():
    for exp_type, exp_names in experiments.items():
        all_models.add(model)
        if exp_type not in all_experiments:
            all_experiments[exp_type] = set()
        for exp_name in exp_names:
            all_experiments[exp_type].add(exp_name)
        if exp_type == "n_few_shots":
            for v in range(10):
                for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75):
                    all_experiments[exp_type].add(f"v{v}/{n}")


In [ ]:
all_experiments["models-estimation"]

In [ ]:
open_weight_models = []
for model, experiments in model_to_experiments.items():
    for exp_type, exp_names in experiments.items():
        if exp_type == "open-weight-models":
            open_weight_models.append(model)

In [ ]:
api_models = []
for model, experiments in model_to_experiments.items():
    for exp_type, exp_names in experiments.items():
        if exp_type == "api-models":
            api_models.append(model)
api_models

In [ ]:
cost_estimation_models = []
for model, experiments in model_to_experiments.items():
    for exp_type, exp_names in experiments.items():
        if exp_type in ["models-estimation", "thinking"]:
            cost_estimation_models.append((model, exp_names))
cost_estimation_models

In [ ]:
# datasets_map = {
#     "anatomy": ["human-mouse"],
#     # "largebio": ["fma-nci", "fma-snomed"][:1],
#     "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"][-2:-1],
# }
# models = [
#     # "mistral.mistral-small-2402-v1:0",
#     # "mistral.mistral-large-2402-v1:0",
#     # "gemini-2.0-flash-lite",
#     # "gemini-2.5-flash-lite",
#     # "gemini-2.5-flash",
#     # "gemini-2.5-flash-pro",
#     "google|gemma-3-4b-it",
#     # "google/gemma-3-12b-it",
#     # "gemini-2.5-pro",
#     # "gpt-5-nano",
#     # "us.meta.llama3-2-3b-instruct-v1:0",
#     # "anthropic.claude-instant-v1",
#     # "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
#     # "meta.llama3-70b-instruct-v1:0",
# ]
# experiments_subtypes = {"syspromts": ["none", "basse", "natural_language", "ontology_aware", "synonym_aware"]}
# experiments_subtypes = {"determinism": ["0", "1", "2", "3"]}
# experiments_subtypes = {
#     "n_few_shots": [f"v{v}/{n}" for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)[:12] for v in range(2)]
# }
# # experiments_subtypes = {"open-weight-models": ["bedrock", "meta.llama3-70b-instruct-v1:0"]}
# # experiments_subtypes = {"reasoning": ["no-reasoning", "with-reasoning"]}
# experiments_subtypes = {"leak_check": ["uri_0", "uri_1"]}

In [ ]:
leak_experiments_subtypes = {"leak_check": ["uri_0", "uri_1"]}
leak_models = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-flash",
    "gemini-1.5-flash-lite",
    "gemini-2.0-flash-lite",
    "gemini-2.5-flash-lite",
    "mistralai|mistral-small-3.2-24b-instruct",
    "meta-llama|llama-3.1-70b-instruct",
]

leak_tasks = ["human-mouse"]

leak_metrics_df = get_experiments_metrics(
    task_names=leak_tasks, models=leak_models, experiments_subtypes=leak_experiments_subtypes
)

In [ ]:
all_df = get_experiments_metrics(task_names=all_tasks, models=list(all_models), experiments_subtypes=all_experiments)
all_df.to_csv(RUN_DIR / "all_experiments_metrics_0.csv", index=False, sep="\t")

In [ ]:
models_experiments_subtypes = {"open-weight-models": open_weight_models, "api-models": api_models}
models = open_weight_models + api_models
# models = ["openai|gpt-oss-120b", "mistralai|mistral-large-2512", "x-ai|grok-4.1-fast"]

# tasks = ["human-mouse"]
tasks = all_tasks

models_metrics_df = get_experiments_metrics(
    task_names=tasks, models=models, experiments_subtypes=models_experiments_subtypes
)

In [ ]:
models_metrics_df.sort_values(["Youden's index"], ascending=False)

In [ ]:
# models_metrics_df.to_csv(RUN_DIR / "open_weight_models_metrics.csv", index=False, sep="\t")

In [ ]:
fdf = models_metrics_df.copy()
fdf = fdf[fdf["SubSet"] == "human-mouse"]

# group by model and select the best experiment for each model
best_experiments_df = fdf.loc[fdf.groupby("Model")["Youden's index"].idxmax()]

best_experiments_df = best_experiments_df.sort_values(
    by="Youden's index", ascending=False
)
best_experiments_df = best_experiments_df.head(len(best_experiments_df) - 1)

plt.figure(figsize=(10, 6))
sns.barplot(data=best_experiments_df, x="Model", y="Youden's index", palette="Set2")
plt.title("Best Youden's Index for Each Model on Human-Mouse Task")
plt.xlabel("Model")
plt.ylabel("Best Youden's Index")
plt.xticks(rotation=-90)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

fdf = models_metrics_df.copy()
fdf = fdf[fdf["SubSet"] == "human-mouse"]

# group by model and select the best experiment for each model
best_experiments_df = fdf.loc[fdf.groupby("Model")["Youden's index"].idxmax()]

best_experiments_df = best_experiments_df.sort_values(
    by="Youden's index", ascending=False
)
best_experiments_df = best_experiments_df.head(len(best_experiments_df) - 1)

# plot
fig = px.bar(
    best_experiments_df,
    x="Model",
    y="Youden's index",
    title="Best Youden's Index for Each Model on Human-Mouse Task",
)

fig.update_layout(
    xaxis_title="Model",
    yaxis_title="Best Youden's Index",
    xaxis_tickangle=-90,
)

fig.show()

In [ ]:
thinking_experiments_subtypes = {
    "thinking": [
        *[f"thinking_budget={x}" for x in [512, 128]],
        # *[f"reasoning_effort={x}" for x in ["minimal", "low", "medium", "high"]],
    ]
}
thinking_models = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "google|gemini-2.5-pro"]
thinking_tasks = all_tasks

thinking_metrics_df = get_experiments_metrics(
    task_names=thinking_tasks, models=thinking_models, experiments_subtypes=thinking_experiments_subtypes
)

In [ ]:
import pandas as pd
import plotly.express as px

# Combine data
df = pd.concat(
    [
        models_metrics_df,
        thinking_metrics_df,
        few_shots_metrics_df,
        metrics_df,
    ],
    ignore_index=True,
)
# only tose models that have more than 1 experiment
df = df.groupby("Model").filter(lambda x: x["SubSet"].nunique() > 1)
df["Task"] = df["SubSet"]

METRIC = "Youden's index"

best_model_df = df.groupby(["Task", "Model"], as_index=False)[METRIC].max()

best_per_task = df.groupby("Task", as_index=False)[METRIC].max()
best_per_task["Model"] = "Best"

avg_per_task = best_model_df.groupby("Task", as_index=False)[METRIC].mean()
avg_per_task["Model"] = "Average"

plot_df = pd.concat([best_model_df, best_per_task, avg_per_task], ignore_index=True)

task_order = list(df["Task"].unique())
plot_df["Task"] = pd.Categorical(plot_df["Task"], categories=task_order, ordered=True)

import plotly.colors as pc

base_colors = pc.qualitative.Plotly
unique_models = plot_df["Model"].unique()
color_map = {m: base_colors[i % len(base_colors)] for i, m in enumerate(unique_models)}
color_map["Best"] = "black"  # treat as another "model" visually
color_map["Average"] = "gray"

# Plot
fig = px.bar(
    plot_df,
    x="Task",
    y=METRIC,
    color="Model",
    barmode="group",
    text_auto=".3f",
    color_discrete_map=color_map,
    title=f"{METRIC} per Task: Models + Best + Average",
)

fig.update_layout(
    xaxis_title="Task",
    yaxis_title=METRIC,
    yaxis=dict(range=[0, 1]),
    legend_title="Model",
)

fig.show()
fig.write_html(OUTPUTS_DIR / "selected_models_comparison.html")

In [ ]:
import pandas as pd
import plotly.express as px

# Combine metrics
df = pd.concat(
    [
        models_metrics_df,
        thinking_metrics_df,
        few_shots_metrics_df,
        metrics_df,
    ],
    ignore_index=True,
)
df = df.groupby("Model").filter(lambda x: x["SubSet"].nunique() > 4)
df["Task"] = df["SubSet"]
METRIC = "Youden's index"

# ---- Step 1: Filter configs with most tasks per model ----
config_cols = ["Prompt", "ExperimentType", "ExpName"]
config_task_counts = df.groupby(["Model"] + config_cols)["Task"].nunique().reset_index(name="TaskCount")
max_task_count = config_task_counts.groupby("Model")["TaskCount"].transform("max")
best_coverage_configs = config_task_counts[config_task_counts["TaskCount"] == max_task_count]

# ---- Step 2: Compute average metric across tasks for these configs ----
model_config_mean = (
    df.merge(best_coverage_configs, on=["Model"] + config_cols)
    .groupby(["Model"] + config_cols, as_index=False)[METRIC]
    .mean()
)

# ---- Step 3: Select best configuration per model (highest average Youden) ----
best_configs = model_config_mean.loc[model_config_mean.groupby("Model")[METRIC].idxmax()].reset_index(drop=True)

# ---- Step 4: Keep only rows from the best config per model ----
best_model_df = pd.merge(df, best_configs[["Model"] + config_cols], on=["Model"] + config_cols, how="inner")

# ---- Step 5: Best metric per Task × Model (filtered configs) ----
best_model_df = best_model_df.groupby(["Task", "Model"], as_index=False)[METRIC].max()

# ---- Step 6: Compute Best per task on the full dataset ----
best_per_task = df.groupby("Task", as_index=False)[METRIC].max()
best_per_task["Model"] = "Best"

# ---- Step 7: Compute Average per model across all tasks (separate task) ----
avg_per_model = best_model_df.groupby("Model")[METRIC].mean().reset_index()
avg_per_model["Task"] = "Average"

# ---- Step 8: Compute Average of Best (mean of all per-task Bests) ----
avg_of_best = pd.DataFrame({"Task": ["Average"], "Model": ["Average of Best"], METRIC: [best_per_task[METRIC].mean()]})

# ---- Step 9: Combine all for plotting ----
plot_df = pd.concat([best_model_df, best_per_task, avg_per_model, avg_of_best], ignore_index=True)

# ---- Step 10: Order tasks ----
task_order = list(best_model_df["Task"].unique()) + ["Average"]
plot_df["Task"] = pd.Categorical(plot_df["Task"], categories=task_order, ordered=True)

# ---- Step 11: Color mapping ----
import plotly.colors as pc

base_colors = pc.qualitative.Plotly
unique_models = plot_df["Model"].unique()
color_map = {
    m: base_colors[i % len(base_colors)] for i, m in enumerate(unique_models) if m not in ["Best", "Average of Best"]
}
color_map["Best"] = "black"
color_map["Average of Best"] = "darkred"  # highlight Average-of-Best

# ---- Step 12: Plot ----
fig = px.bar(
    plot_df,
    x="Task",
    y=METRIC,
    color="Model",
    barmode="group",
    text_auto=".3f",
    color_discrete_map=color_map,
    title=f"{METRIC} per Task: Models (Best Configs) + Best + Average + Average-of-Best",
)

fig.update_layout(
    xaxis_title="Task",
    yaxis_title=METRIC,
    yaxis=dict(range=[0, 1]),
    legend_title="Model",
)

fig.show()

# ---- Step 13: Save the plot as HTML ----
fig.write_html(OUTPUTS_DIR / "best_configs_comparison_1.html")

In [ ]:
import pandas as pd
import numpy as np

# ---- Step 1: Use filtered best_model_df (best config per model) ----
# best_model_df has columns: ["Task", "Model", METRIC]

# Pivot so rows=Model, columns=Task
task_matrix = best_model_df.pivot(index="Model", columns="Task", values=METRIC)

# ---- Step 2: Compute average performance per model ----
avg_per_model = task_matrix.mean(axis=1)

# ---- Step 3: Compute correlation per task with avg_per_model ----
correlations = task_matrix.corrwith(avg_per_model)

# ---- Step 4: Sort tasks by correlation ----
correlation_df = correlations.reset_index()
correlation_df.columns = ["Task", "CorrelationWithAverage"]
correlation_df = correlation_df.sort_values(by="CorrelationWithAverage", ascending=False)

print("Tasks most correlated with average performance:")
print(correlation_df)


In [ ]:
models_metrics_df.sort_values(["Youden's index"], ascending=False).head(10)

In [ ]:
df = models_metrics_df
# rows = 

df = df[df["SubSet"] == "human-mouse"]
df = df.groupby(["Model", "ExperimentType", "ExpName"]).max().reset_index()
selected = df.sort_values(by="Youden's index", ascending=False).head(5)

# plot horizontal bar plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    ax=ax,
    data=selected,
    y="Model",
    x="Youden's index",
    hue="ExpName",
    palette="viridis",
)
ax.set_title("Top 5 Models by Youden's Index")
ax.set_xlabel("Youden's Index")
ax.set_ylabel("Model")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# results_df.to_csv(METRICS_DIR / "open-source-models-2025-10-05.csv", index=False)

In [ ]:
few_shots_experiments_subtypes = {
    "n_few_shots": [f"v{v}/{n}" for n in (0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75) for v in range(5)]
}
few_shot_models = [
    "anthropic.claude-3-sonnet-20240229-v1:0",
    "anthropic.claude-instant-v1",
    "deepseek|deepseek-chat-v3-0324",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
    "gemini-3-pro-preview",
    "google|gemma-3-12b-it",
    "google|gemma-3-27b-it",
    "google|gemma-3-4b-it",
    "gpt-4.1-nano",
    "gpt-5-nano",
    "meta.llama3-70b-instruct-v1:0",
    "meta-llama|llama-4-scout",
    "mistralai",
    "mistralai|ministral-14b-2512",
    "mistralai|ministral-3b-2512",
    "mistralai|mistral-large-2512",
    "mistralai|mistral-large-2512meta-llama|llama-4-scout",
    "mistralai|mistral-nemo",
    "mistral.mistral-large-2402-v1:0",
    "mistral.mistral-small-2402-v1:0",
    "openai|gpt-5-nano",
    "openai|gpt-oss-120b",
    "qwen3-1.7B",
    "qwen3-8B",
    "qwen|qwen3-8b'",
    "qwen|qwen3-next-80b-a3b-instruct",
    "qwen|qwen3-vl-8b-instruct",
    "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    "us.meta.llama3-2-3b-instruct-v1:0",
    "x-ai|grok-4.1-fast",
]
few_shot_tasks = [
    "ncit-doid",
    "omim-ordo",
    "snomed-fma.body",
    "snomed-ncit.neoplas",
    "snomed-ncit.pharm",
    "fma-nci",
    "fma-snomed",
    "snomed-nci",
    "human-mouse",
]
few_shots_metrics_df = get_experiments_metrics(
    task_names=few_shot_tasks, models=few_shot_models, experiments_subtypes=few_shots_experiments_subtypes
)

In [ ]:
few_shots_metrics_df.groupby("SubSet").apply(lambda x: x.sort_values("Youden's index", ascending=False).head(7))

In [ ]:
gemini_pro_df = few_shots_metrics_df[few_shots_metrics_df["Model"].isin(["gemini-2.5-pro", "gemini-3-pro-preview"])]
gemini_pro_df.groupby("SubSet").apply(lambda x: x.sort_values("Youden's index", ascending=False).head(7))

In [ ]:
gemini_pro_df = few_shots_metrics_df[few_shots_metrics_df["Model"].isin(["gemini-2.5-pro", "gemini-3-pro-preview"])]
gemini_pro_df = gemini_pro_df.groupby(["Model", "SubSet"]).max().reset_index()

# plot horizontal bar plot with the best results for each model
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    ax=ax,
    data=gemini_pro_df,
    y="SubSet",
    x="Youden's index",
    hue="Model",
    palette="viridis",
)
# ax.set_title("Gemini Pro Models Performance by Matching Task")
ax.set_xlabel("Youden's Index")
# ax.set_ylabel("Matching Task")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
reasoning_experiments_subtypes = {"reasoning": ["no-reasoning", "with-reasoning"]}
reasoning_models = ["gemini-2.0-flash-lite"]
reasoning_tasks = ["human-mouse", "snomed-ncit.pharm"]

reasoning_metrics_df = get_experiments_metrics(
    task_names=reasoning_tasks, models=reasoning_models, experiments_subtypes=reasoning_experiments_subtypes
)

In [ ]:
thinking_experiments_subtypes = {
    "thinking": [
        *[f"thinking_budget={x}" for x in [0, 512, 1024]],
        *[f"reasoning_effort={x}" for x in ["minimal", "low", "medium", "high"]],
    ]
}
thinking_models = ["gemini-2.5-flash-lite", "openai|gpt-5-nano"]
thinking_tasks = ["human-mouse", "snomed-ncit.pharm"]

thinking_metrics_df = get_experiments_metrics(
    task_names=thinking_tasks, models=thinking_models, experiments_subtypes=thinking_experiments_subtypes
)

In [ ]:
thinking_metrics_df.groupby("SubSet").apply(lambda x: x.sort_values("Youden's index", ascending=False).head(3))

In [ ]:
model = "openai|gpt-5-nano"
prompt = "direct_entity"
sub_df = thinking_metrics_df[(thinking_metrics_df["Model"] == model) & (thinking_metrics_df["Prompt"] == prompt)]
sub_df

# plot on one figure two sublots as bar plots the Youden's index for differnt SubSets, one subsset on one subplot
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, subset in zip(axes, thinking_tasks):
    subset_df = sub_df[sub_df["SubSet"] == subset]
    sns.barplot(
        ax=ax,
        data=subset_df,
        x="ExpName",
        y="Youden's index",
        palette="viridis",
    )
    ax.set_title(f"Youden's Index for {subset}")
    ax.set_xlabel("Experiment")
    ax.set_ylabel("Youden's Index")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

In [ ]:
thinking_metrics_df

In [ ]:
for model in few_shot_models:
    plot_nfew_shots_analysis_mean(few_shots_metrics_df, model=model)
# plot_nfew_shots_analysis_mean(few_shots_metrics_df, model="gemini-2.5-flash-lite")

In [ ]:
plot_nfew_shots_analysis_mean(results_df[results_df["Model"] == "gemini-2.5-flash-lite"])

In [ ]:
metrics_df = results_df.copy()
metric = "Youden's index"

experiment_col, left_col, right_col = "Matching Task", "Prompt", "ExpName"

for metric in ["Youden's index", "F1 Score"]:
    plot_detailed_analysis(
        metrics_df=metrics_df,
        metric=metric,
        run_col_name="LLM-Prompt",
        experiment_col=experiment_col,
        left_col=left_col,
        right_col=right_col,
        left_col_name=left_col,
        right_col_name=right_col,
        run_separator="-",
        precision=4,
        subfolder_name=f"{next(iter(experiments_subtypes.keys()))}/{metric}",
        save_plots=False,
    )
    plt.show()

## Error rates

In [ ]:
outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
    "google|gemma3-4b-it",
]

# few-shot experiments
experiment_type = "n_few_shots"
experiments = [
    f"v{version}/{n}"
    for version in range(5)
    for n in [
        0,
        1,
        2,
        3,
        5,
        8,
        11,
        15,
        20,
        26,
        33,
        41,
        # 50,
        # 62,
        # 75,
    ]
]

records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for exp in experiments:
                run_subdir = f"{dataset}/{subset}/{model}/{experiment_type}/{exp}"
                path = outputs_dir / run_subdir / "results_reduced.csv"

                if not path.exists() or os.path.getsize(path) == 0:
                    # file missing
                    # records.append(
                    #     {
                    #         "Dataset": dataset,
                    #         "Subset": subset,
                    #         "Model": model,
                    #         "Experiment": exp,
                    #         "Prompt": "NO_FILE",
                    #         "Exists": False,
                    #     }
                    # )
                    continue

                try:
                    df_local = pd.read_csv(path)
                except Exception as e:
                    continue

                if "Prompt" not in df_local.columns:
                    continue

                for prompt in df_local["Prompt"].unique():
                    records.append(
                        {
                            "Dataset": dataset,
                            "Subset": subset,
                            "Model": model,
                            "Experiment": exp,
                            "Prompt": prompt,
                            "Exists": True,
                        }
                    )

df = pd.DataFrame(records)
df["TaskPrompt"] = df["Dataset"] + "/" + df["Subset"] + " | " + df["Prompt"]

sns.set_theme(style="whitegrid")

for model in df["Model"].unique():
    model_df = df[df["Model"] == model]

    pivot = model_df.pivot_table(
        index="TaskPrompt", columns="Experiment", values="Exists", aggfunc="first", fill_value=False
    )
    # print missing experiments in format task, prompt, n_few_shots_version/number from pivot table
    missing = pivot[pivot == False].stack().reset_index()
    for _, row in missing.iterrows():
        print(f"{row['TaskPrompt']}, {row['Experiment']}")

    plt.figure(figsize=(16, max(6, 0.3 * len(pivot))))
    sns.heatmap(
        pivot,
        cmap=["#ffcccc", "#c6efce"],  # red = missing, green = exists
        cbar=False,
        linewidths=0.4,
        linecolor="gray",
    )

    plt.title(f"Missing Experiments for {model} (Grouped by Task & Prompt)", fontsize=14)
    plt.xlabel("Few-shot version / number")
    plt.ylabel("Task / Prompt")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    "gemini-2.0-flash-lite",
    # "mistral.mistral-large-2402-v1:0",
]

experiment_type = "n_few_shots"
few_shot_versions = ["v0", "v1", "v2", "v3", "v4"]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]

records = []

# === COLLECT DATA ===
for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for v, n in product(few_shot_versions, few_shot_numbers):
                run_dir = outputs_dir / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob("*.csv"):
                    if csv_file.name == "results_reduced.csv":
                        continue

                    prompt_name = csv_file.stem.replace("_results_reduced", "")
                    if f"{model}_" in prompt_name:
                        prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        print(f"Error reading {csv_file}: {e}")
                        continue

                    if "Prediction" not in df.columns:
                        continue

                    total = len(df)
                    if total == 0:
                        continue

                    false_count = (df["Prediction"].astype(str).str.lower() == "error").sum()
                    error_rate = false_count / total

                    if error_rate == 0:
                        continue

                    records.append(
                        {
                            "Task": f"{dataset}/{subset}",
                            "Model": model,
                            "Prompt": prompt_name,
                            "FewShot": f"{v}/{n}",
                            "ErrorRate": error_rate,
                        }
                    )

results_df = pd.DataFrame(records)
if results_df.empty:
    raise ValueError("No valid few-shot experiments with errors found!")

results_df["Label"] = results_df["Task"] + "/" + results_df["Prompt"] + "/" + results_df["FewShot"]

sns.set(style="whitegrid")

plt.figure(figsize=(max(14, len(results_df) * 0.25), 7))
sns.barplot(
    data=results_df.sort_values("ErrorRate", ascending=False),
    x="Label",
    y="ErrorRate",
    hue="Model",
    palette="Reds",
)

plt.title("Error Rate per Task / Prompt / Few-Shot", fontsize=15)
plt.ylabel("Error Rate (Error / Total)")
plt.xlabel("Task / Prompt / Few-Shot")
plt.xticks(rotation=90, ha="center", fontsize=8)
plt.legend(title="Model")
plt.tight_layout()
plt.yscale("log")
plt.grid(axis="y", alpha=0.3)

In [ ]:
outputs_dir = Path("./outputs")

datasets_map = {
    "anatomy": ["human-mouse"],
    "largebio": ["fma-nci", "fma-snomed"],
    "bioml-2024": ["omim-ordo", "ncit-doid", "snomed-ncit.pharm", "snomed-fma.body"],
}

models = [
    # "mistral.mistral-large-2402-v1:0",
    "gemini-2.0-flash-lite",
]

experiment_type = "n_few_shots"
few_shot_versions = ["v0", "v1", "v2", "v3", "v4"]
few_shot_numbers = [0, 1, 2, 3, 5, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75]

records = []

for dataset, subsets in datasets_map.items():
    for subset in subsets:
        for model in models:
            for n, v in product(few_shot_numbers, few_shot_versions):
                run_dir = outputs_dir / dataset / subset / model / experiment_type / f"{v}/{n}"
                if not run_dir.exists():
                    continue

                for csv_file in run_dir.glob("*.csv"):
                    if csv_file.name == "results_reduced.csv":
                        continue

                    prompt_name = csv_file.stem.replace("_results_reduced", "")
                    prompt_name = prompt_name.split(f"{model}_")[-1]

                    try:
                        df = pd.read_csv(csv_file)
                    except Exception as e:
                        continue

                    if "Prediction" not in df.columns:
                        continue

                    counts = df["Prediction"].value_counts().to_dict()
                    total = len(df)

                    for pred_label, count in counts.items():
                        records.append(
                            {
                                "Dataset": dataset,
                                "Subset": subset,
                                "Task": f"{dataset}/{subset}",
                                "Model": model,
                                "FewShot": f"{v}/{n}",
                                "Prompt": prompt_name,
                                "Prediction": str(pred_label),
                                "Count": count,
                                "Total": total,
                            }
                        )

results_df = pd.DataFrame(records)
if results_df.empty:
    raise ValueError("No CSV data found — check paths and filenames!")

results_df["Proportion"] = results_df["Count"] / results_df["Total"]

sns.set(style="whitegrid")

for (task, prompt), group_df in results_df.groupby(["Task", "Prompt"]):
    plt.figure(figsize=(14, 4))
    # group_df = group_df[group_df["Prediction"].isin(["error", "no_match", "match"])]
    sns.barplot(
        data=group_df,
        x="FewShot",
        y="Proportion",
        hue="Prediction",
        palette="Set2",
    )
    plt.title(f"{task} — {prompt}")
    plt.ylabel("Proportion of Predictions")
    plt.xlabel("Few-shot version / number")
    plt.xticks(rotation=90, ha="right")
    plt.legend(title="Prediction")
    plt.tight_layout()
    plt.show()


## Radar Charts

In [ ]:
f1_oracle_df = pd.read_csv(METRICS_DIR / "OASystem_with_oracle_f1.csv", sep='\t')
f1_oracle_df = f1_oracle_df[
    [
        "Task",
        '$OASystem(auto)$',
        # '$OASystem+Or^{GF2.0}$',
        '$OASystem+Or^{0}$',
        # '$OASystem+Or^{10}$',
        '$OASystem+Or^{20}$',
        '$OASystem+Or^{30}$',
        '$OASystem+Or^{LLM}$',
    ]
]
# renale OASystem to LogMap in all columns except Task
f1_oracle_df = f1_oracle_df.rename(columns={"$OASystem+Or^{LLM}$": "$LogMap+Or^{LLM}_{GF2.5}$"})
f1_oracle_df = f1_oracle_df.rename(columns=lambda x: x.replace("OASystem", "LogMap"))
labels = f1_oracle_df.columns[1:]
f1_oracle_df

In [ ]:
oracle_df = f1_oracle_df.copy()

categories1= oracle_df["Task"].tolist()
values_list1 = oracle_df.iloc[:, 1:].to_numpy().T.tolist()

colors = ["blue", "green", "purple", "red", "black"]
linestyles = ["solid", "dashed", "dotted", "dashdot", (0, (5, 2))]


plt.rcParams["figure.dpi"] =444

fig, axs = plt.subplots(1, 1, subplot_kw={"polar": True}, figsize=(8, 8))
make_radar_chart(axs, categories1, values_list1, labels, colors, linestyles, (0.5, 0.99), "", scale=1.03)

handles, _ = axs.get_legend_handles_labels()
fig.legend(handles, labels,loc="lower center", ncol=3, fontsize=10)

# fig.suptitle("Comparison against Oracles with different error rates", fontsize=14)
# plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.tight_layout()
# plt.subplots_adjust(hspace=0.3)

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_radar_chart_all.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(save_path, bbox_inches="tight")

In [ ]:
idx = 4
oracle_df = f1_oracle_df.copy()

categories1, categories2 = oracle_df["Task"].tolist()[:idx], oracle_df["Task"].tolist()[idx:]
values_list1 = oracle_df.iloc[:, 1:].to_numpy()[:idx].T.tolist()
values_list2 = oracle_df.iloc[:, 1:].to_numpy()[idx:].T.tolist()

colors = ["blue", "green", "purple", "red", "black"]
linestyles = ["solid", "dashed", "dotted", "dashdot", (0, (5, 2))]


plt.rcParams["figure.dpi"] = 444

fig, axs = plt.subplots(1, 2, subplot_kw={"polar": True}, figsize=(14, 7))
axs = axs.flatten()

make_radar_chart(axs[0], categories1, values_list1, labels, colors, linestyles, (0.7, 0.9), "", scale=1.03)
make_radar_chart(axs[1], categories2, values_list2, labels, colors, linestyles, (0.55, 0.72), "", scale=1.05)

handles, _ = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10)

# fig.suptitle("Comparison against Oracles with different error rates", fontsize=14)
plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_radar_chart_verical.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, bbox_inches="tight")

In [ ]:
exp_oracle_df = pd.read_csv(METRICS_DIR / "OASystem_with_oracle_f1.csv", sep="\t")

for col in exp_oracle_df.columns[:0:-1]:
    exp_oracle_df[col] -= exp_oracle_df.iloc[:, 1]
exp_oracle_df

In [ ]:
plt.figure(figsize=(12, 6))

for col in exp_oracle_df.columns[1:]:
    if "LLM}$" in col:
        plt.plot(exp_oracle_df["Task"], exp_oracle_df[col], label=col, color="black", marker="o")
    else:
        plt.plot(exp_oracle_df["Task"], exp_oracle_df[col], label=col, linestyle="--", marker="o")

plt.xticks(rotation=20, ha="center")
plt.ylabel(r"$\Delta$Score")
plt.title("Model Comparison across Tasks")
plt.legend()
plt.grid(True)
plt.tight_layout()

save_path = METRICS_DIR / datetime.today().strftime("%Y-%m-%d") / "OASystem" / "OA_with_oracle_line_chart.png"
# save_path.parent.mkdir(parents=True, exist_ok=True)
# plt.savefig(save_path, bbox_inches="tight")